#  Problem Understanding

Account **ABC** has a total asset value of **$100,000**, fully vested, distributed across five securities: **IBM, MSFT, ORCL, AAPL, HD**.

For each security, the system is provided:
- **Target %** — desired allocation of total assets.
- **Current %** — current allocation of total assets.
- **Target Variance** — difference between target and current allocation.
  - Negative variance → underweight → **buy**.
  - Positive variance → overweight → **sell**.
- **Unit Price** — price per share.

The application must compute the **number of shares to buy or sell** for each security so that the portfolio is rebalanced and all securities match their **target %**, resulting in **zero target variance**.

---

# Goal

The goal is to:
- Interpret the rebalancing logic using target %, current %, total asset value, and unit price.
- Calculate the **dollar adjustment** required for each security.
- Convert the dollar adjustment into **number of shares to buy or sell**.
- Ensure that after applying these trades, each security’s allocation equals its target %, achieving **zero variance**.
- Validate correctness through manual and automated tests.

---

# Rebalancing Logic (How to reach zero target variance)

For each security:

####1.Compute target value  

        target_value = (target_pct / 100) * total_assets

####2.Compute current value

        current_value = (current_pct / 100) * total_assets

####3.Dollar difference

     dollar_diff = target_value - current_value

####4.Variance percentage

      variance_pct = current_pct - target_pct

If variance_pct < 0 → buy.

If variance_pct>  0 → sell.

####5.  Raw share count  

shares_raw = dollar_diff / unit_price

####6. Rounded Share Count

shares_rounded = round(shares_raw)

####7.Final Action
if shares_rounded > 0:  action = "BUY"
elif shares_rounded < 0: action = "SELL"
else:                    action = "HOLD"



### Example Walkthrough For IBM Security:

Given:
- target_pct = 20%
- current_pct = 10%
- total_assets = 100,000
- unit_price = 150


### 1. Target value
target_value = (20 / 100) * 100000
target_value = 0.20 * 100000
target_value = 20000
###2.Current value
current_value = (10 / 100) * 100000
current_value = 0.10 * 100000
current_value = 10000
###3.Dollar difference
dollar_diff = target_value - current_value
dollar_diff = 20000 - 10000
dollar_diff = 10000
Positive → IBM is **underweight** → needs **BUY**.
###4.Variance percentage
variance_pct = current_pct - target_pct
variance_pct = 10 - 20
variance_pct = -10
### 5. Raw share count
shares_raw = dollar_diff / unit_price
shares_raw = 10000 / 150
shares_raw = 66.6667
### 6. Rounded share count
Whole shares only.
shares_rounded = round(66.6667)
shares_rounded = 67



---

## Final IBM Output

| Field            | Value      |
|------------------|------------|
| target_value     | 20,000     |
| current_value    | 10,000     |
| dollar_diff      | +10,000    |
| variance_pct     | -10        |
| shares_raw       | 66.6667    |
| shares_rounded   | 67         |
| action           | BUY        |

IBM needs **67 shares BUY** to reach its target allocation.





## The business question this notebook answers

> **What do you have to do to get to zero target variance?**

**Answer, for Account ABC's baseline portfolio ($100,000 total assets):**

| Security | Target % | Current % | Variance | Output - Number of shares to buy/sell |
|---|---|---|---|---|
| IBM  | 20% | 10% | -10% (underweight) | **BUY 67 shares** |
| MSFT | 20% | 20% |  0%                | 0 - HOLD |
| ORCL | 20% | 30% | +10% (overweight)  | **SELL 45 shares** |
| AAPL | 20% | 20% |  0%                | 0 - HOLD |
| HD   | 20% | 20% |  0%                | 0 - HOLD |

Buying 67 shares of IBM (~$10,000) and selling 45 shares of ORCL (~$10,000)
brings every security's current% back in line with its 20% target — every
variance becomes exactly 0%. MSFT, AAPL, and HD are already on target and
need no trade.

This exact result is computed programmatically two cells below (Section 2's
sanity check) and is proven correct by the automated test
`test_TC01_baseline_happy_path` in Section 4 — this section states the
answer in plain business terms; the code and tests below verify it.

####Note:

This notebook :--> writes the reference implementation to
`rebalancer.py`, then runs a `pytest` suite against it via `ipytest` (which
lets standard pytest tests run inside a Jupyter/Colab notebook cell).

Companion document: **manual_test_cases.xlsx** — the manual test case sheet is attached as part of the provided github link.
Every automated test below is tagged with the Manual Test Case ID (TC-xx)
it automates, so coverage is traceable in both directions.

## 1. Environment setup

Run this once per Colab session.

In [8]:
!pip install -q pytest ipytest

In [9]:
import ipytest
ipytest.autoconfig()

#2. System under test

Written to `rebalancer.py` so it behaves like a real module being tested
(and so it can be swapped out for the actual application's module/API in a
real project — only this cell would need to change).


**Scope note:** the 7 assumptions listed down below under rebalancer.py are specific to the `rebalancer.py`
calculation code shown below — they're what its `raise ValueError` checks
and rounding rule actually enforce. Broader business/scope assumptions for
the overall project (partial vesting, the API/UI layers, transaction costs,
single account/currency, etc.) are documented separately in
`ASSUMPTIONS.md`, since this code doesn't implement or enforce those at
all.

### Purpose Of rebalancer.py

The purpose of `rebalancer.py` is to provide a clear, testable, and standalone
implementation of the portfolio rebalancing logic used in the Account ABC
scenario. Instead of keeping formulas scattered across a spreadsheet or
manual notes, the rebalancing rules are captured in a single Python module
that behaves exactly like a real production component.

This allows us to:
- run automated tests against a real function,
- validate every step of the calculation,
- document assumptions explicitly,
- demonstrate how the rebalancing engine should behave,
- and ensure the logic is correct, repeatable, and easy to maintain.

In short, `rebalancer.py` is the **System Under Test (SUT)** for both manual and
automated test cases.

In [10]:
%%writefile rebalancer.py
"""
rebalancer.py

Reference implementation of the portfolio rebalancing calculation shown in
the "Account ABC" spreadsheet. This is the System Under Test (SUT) for the
manual and automated test suites in this project.

BUSINESS RULE BEING IMPLEMENTED
--------------------------------
For each security in an account:

    target_value  = target_pct  / 100 * total_assets
    current_value = current_pct / 100 * total_assets
    dollar_diff   = target_value - current_value
    variance_pct  = current_pct - target_pct        (matches the sheet's
                                                       "target Variance" column:
                                                       negative = underweight = BUY,
                                                       positive = overweight  = SELL)
    shares_raw    = dollar_diff / unit_price
    shares_rounded = round(shares_raw) to the nearest whole share,
                     sign preserved (standard round-half-up on the magnitude)

    action = "BUY"  if shares_rounded > 0
             "SELL" if shares_rounded < 0
             "HOLD" if shares_rounded == 0

ASSUMPTIONS DOCUMENTED FOR TESTING (confirm with the business/product owner
before treating these as final — they are exactly the kind of ambiguity a
tester should flag rather than silently assume):

1. Fractional shares are NOT tradable. Every result is rounded to a whole
   share using standard round-half-up on the absolute value, then the sign
   (buy vs. sell) is re-applied. E.g. 66.67 -> 67, -45.45 -> -45... wait,
   45.45 rounds to 45 (0.45 rounds down), while 66.67 rounds to 67
   (0.67 rounds up) — see the rounding helper below for the exact rule.
2. total_assets must be strictly positive. Zero or negative total assets is
   treated as invalid input (raises ValueError) rather than silently
   producing 0/0 or infinite share counts.
3. unit_price must be strictly positive for the same reason (avoids a
   ZeroDivisionError / negative-share nonsense from a bad price feed).
4. target_pct and current_pct must be >= 0. Negative percentages are
   rejected as bad data.
5. The engine does NOT require sum(target_pct) == 100 or
   sum(current_pct) == 100 across the account — it computes each security
   independently, exactly as the spreadsheet does. A separate validation
   helper (validate_allocations) is provided so a caller/tester can check
   that invariant explicitly and decide what to do about it.
6. No minimum trade size / no cash-sufficiency check is enforced. A real
   trading system would likely also verify total buys can be funded by
   total sells (or available cash) before submitting orders — this
   reference implementation intentionally leaves that out so tests can
   demonstrate it's a gap (see TC-11 / TC-15 in the manual test cases).
7. VESTING (see TC-25): target_pct/current_pct are applied against the
   VESTED (investable) portion of total_assets, not the full account
   value. investable_base = total_assets * vested_pct / 100. Passing the
   default vested_pct=100.0 reproduces every prior calculation exactly
   (investable_base == total_assets), so this is backward compatible with
   TC-01 through TC-16/19/20. This is a design DECISION made to make
   TC-25 automatable, not a confirmed product requirement — flag it for
   business sign-off before relying on it.
"""

from dataclasses import dataclass, asdict
from typing import List
import math


@dataclass
class SecurityInput:
    symbol: str
    target_pct: float
    current_pct: float
    unit_price: float


@dataclass
class RebalanceResult:
    symbol: str
    target_pct: float
    current_pct: float
    variance_pct: float
    unit_price: float
    dollar_diff: float
    shares_raw: float
    shares_rounded: int
    action: str  # "BUY", "SELL", "HOLD"
    vested_pct: float = 100.0
    investable_base: float = 0.0

    def as_dict(self):
        return asdict(self)


def _round_shares(raw_shares: float) -> int:
    """Round-half-up on magnitude, sign preserved. 0 stays 0."""
    if raw_shares == 0:
        return 0
    sign = 1 if raw_shares > 0 else -1
    return sign * int(math.floor(abs(raw_shares) + 0.5))


def validate_allocations(securities: List[SecurityInput], tolerance: float = 0.01):
    """
    Optional data-integrity check: do target_pct values sum to 100%, and do
    current_pct values sum to 100%? Returns a dict of booleans; does not
    raise, so callers/tests can decide how strict to be.
    """
    target_sum = sum(s.target_pct for s in securities)
    current_sum = sum(s.current_pct for s in securities)
    return {
        "target_sum": target_sum,
        "current_sum": current_sum,
        "target_sums_to_100": abs(target_sum - 100) <= tolerance,
        "current_sums_to_100": abs(current_sum - 100) <= tolerance,
    }


def calculate_rebalance(
    securities: List[SecurityInput], total_assets: float, vested_pct: float = 100.0
) -> List[RebalanceResult]:
    if total_assets is None or total_assets <= 0:
        raise ValueError("total_assets must be a positive number")
    if vested_pct is None or not (0 < vested_pct <= 100):
        raise ValueError("vested_pct must be > 0 and <= 100")

    investable_base = total_assets * vested_pct / 100

    if not securities:
        return []

    results = []
    for sec in securities:
        if sec.unit_price is None or sec.unit_price <= 0:
            raise ValueError(f"{sec.symbol}: unit_price must be a positive number")
        if sec.target_pct < 0 or sec.current_pct < 0:
            raise ValueError(f"{sec.symbol}: percentages cannot be negative")

        target_value = sec.target_pct / 100 * investable_base
        current_value = sec.current_pct / 100 * investable_base
        dollar_diff = target_value - current_value
        variance_pct = sec.current_pct - sec.target_pct
        shares_raw = dollar_diff / sec.unit_price
        shares_rounded = _round_shares(shares_raw)

        if shares_rounded > 0:
            action = "BUY"
        elif shares_rounded < 0:
            action = "SELL"
        else:
            action = "HOLD"

        results.append(
            RebalanceResult(
                symbol=sec.symbol,
                target_pct=sec.target_pct,
                current_pct=sec.current_pct,
                variance_pct=round(variance_pct, 6),
                unit_price=sec.unit_price,
                dollar_diff=round(dollar_diff, 6),
                shares_raw=shares_raw,
                shares_rounded=shares_rounded,
                action=action,
                vested_pct=vested_pct,
                investable_base=round(investable_base, 6),
            )
        )
    return results


def calculate_single_line(
    current_pct: float, target_pct: float, unit_price: float, total_assets: float, vested_pct: float = 100.0
) -> float:
    """Convenience wrapper used by parametrized single-line unit tests."""
    result = calculate_rebalance(
        [SecurityInput(symbol="X", target_pct=target_pct, current_pct=current_pct, unit_price=unit_price)],
        total_assets=total_assets,
        vested_pct=vested_pct,
    )
    return result[0].shares_rounded


Overwriting rebalancer.py


In [11]:
from rebalancer import (
    SecurityInput,
    RebalanceResult,
    calculate_rebalance,
    calculate_single_line,
    validate_allocations,
)

# Quick sanity check against the original spreadsheet numbers
demo = calculate_rebalance(
    [
        SecurityInput("IBM", target_pct=20, current_pct=10, unit_price=150),
        SecurityInput("MSFT", target_pct=20, current_pct=20, unit_price=90),
        SecurityInput("ORCL", target_pct=20, current_pct=30, unit_price=220),
        SecurityInput("AAPL", target_pct=20, current_pct=20, unit_price=450),
        SecurityInput("HD", target_pct=20, current_pct=20, unit_price=70),
    ],
    total_assets=100_000,
)
for r in demo:
    print(f"{r.symbol:5s} variance={r.variance_pct:+.1f}%  action={r.action:4s}  shares={r.shares_rounded}")

print()
print("Same baseline, 80% vested ($80,000 investable of $100,000 total):")
vested_demo = calculate_rebalance(
    [SecurityInput("IBM", target_pct=100, current_pct=0, unit_price=150)],
    total_assets=100_000,
    vested_pct=80,
)
for r in vested_demo:
    print(f"  {r.symbol}: investable_base=${r.investable_base:,.0f}  action={r.action}  shares={r.shares_rounded}")

IBM   variance=-10.0%  action=BUY   shares=67
MSFT  variance=+0.0%  action=HOLD  shares=0
ORCL  variance=+10.0%  action=SELL  shares=-45
AAPL  variance=+0.0%  action=HOLD  shares=0
HD    variance=+0.0%  action=HOLD  shares=0

Same baseline, 80% vested ($80,000 investable of $100,000 total):
  IBM: investable_base=$80,000  action=BUY  shares=533


## 3. Manual -> Automated test case traceability

| Manual TC ID | Scenario | Automated test |
|---|---|---|
| TC-01 | Baseline happy path (mixed buy/sell/hold) | `test_TC01_baseline_happy_path` |
| TC-02 | All holdings already at target | `test_TC02_all_on_target_produces_no_trades` |
| TC-03 | Fully underweight single security | `test_TC03_single_security_fully_underweight` |
| TC-04 | Total assets = 0 / negative | `test_TC04_zero_total_assets_raises_value_error`, `test_TC04b_negative_total_assets_raises_value_error` |
| TC-05 | Unit price = 0 | `test_TC05_zero_unit_price_raises_value_error` |
| TC-06 | Negative unit price | `test_TC06_negative_unit_price_raises_value_error` |
| TC-07 | Target% doesn't sum to 100 | `test_TC07_target_percentages_not_summing_to_100_is_flagged_not_fatal` |
| TC-08 | Current% doesn't sum to 100 | `test_TC08_current_percentages_not_summing_to_100_is_flagged` |
| TC-09 | Sub-share variance rounds to 0 | `test_TC09_sub_share_variance_rounds_to_zero` |
| TC-10 | Rounding boundary behavior | `test_TC10_rounding_boundaries` (parametrized) |
| TC-11 | All holdings overweight (no buys) | `test_TC11_all_overweight_produces_only_sells` |
| TC-12 | Negative percentage input | `test_TC12_negative_percentage_raises_value_error` |
| TC-13 | Empty portfolio | `test_TC13_empty_portfolio_returns_empty_list` |
| TC-14 | Large-scale account / precision | `test_TC14_large_scale_account_precision` |
| TC-15 | Rounding drift within tolerance | `test_TC15_rounding_drift_within_tolerance` |
| TC-16 | Variance sign always matches action | `test_variance_sign_matches_action` (parametrized) |
| TC-17, TC-18 | UI/workflow only | Manual only — no automated equivalent (see `manual_test_cases.xlsx`) |
| TC-19 | Full portfolio, every line trades | `test_TC19_full_portfolio_all_lines_trade` |
| TC-20 | Scale invariance ($100K vs $250K) | `test_TC20_scale_invariance` |
| TC-21, TC-22, TC-23, TC-24 | API contract layer | Manual only / Blocked — no live API endpoint exists in this reference project (see `manual_test_cases.xlsx`) |
| TC-25 | Partial vesting reduces investable base | `test_TC25_partial_vesting_reduces_investable_base`, `test_TC25b_...backward_compatible`, `test_TC25c_...invalid_vested_pct` |

## 4. Automated test suite

Standard `pytest` tests — collected and run in-notebook via `ipytest`.

In [12]:
"""
test_rebalancer.py

Automated test suite for rebalancer.py.

Each test is tagged in its docstring with the Manual Test Case ID it
automates (see manual_test_cases.xlsx) so coverage is traceable both ways:
manual case -> automated test, and automated test -> manual case.
"""

import pytest
from rebalancer import (
    SecurityInput,
    calculate_rebalance,
    calculate_single_line,
    validate_allocations,
)


# ---------------------------------------------------------------------------
# Sample data
# ---------------------------------------------------------------------------

def baseline_portfolio():
    return [
        SecurityInput(symbol="IBM", target_pct=20, current_pct=10, unit_price=150),
        SecurityInput(symbol="MSFT", target_pct=20, current_pct=20, unit_price=90),
        SecurityInput(symbol="ORCL", target_pct=20, current_pct=30, unit_price=220),
        SecurityInput(symbol="AAPL", target_pct=20, current_pct=20, unit_price=450),
        SecurityInput(symbol="HD", target_pct=20, current_pct=20, unit_price=70),
    ]


def assert_shares_match(results, expected_shares: dict):
    """
    THE reusable pattern for answering 'Output - Number of shares to
    buy/sell' per security, automatically, regardless of how many
    securities are in the portfolio or what order calculate_rebalance
    returned them in:

    1. Index the results list by symbol (results come back in input order,
       but tests shouldn't depend on that).
    2. Confirm the exact set of symbols matches what was expected (catches
       a security silently missing from the output).
    3. Assert the exact rounded share count for every symbol, one at a time,
       so a failure names precisely which security's output is wrong
       rather than failing the whole test as one opaque blob.

    expected_shares: {"IBM": 67, "ORCL": -45, "MSFT": 0, ...}
    """
    by_symbol = {r.symbol: r for r in results}
    assert set(by_symbol.keys()) == set(expected_shares.keys()), (
        f"Symbol mismatch: got {set(by_symbol.keys())}, expected {set(expected_shares.keys())}"
    )
    for symbol, expected in expected_shares.items():
        actual = by_symbol[symbol].shares_rounded
        assert actual == expected, f"{symbol}: expected {expected} shares, got {actual}"


# ---------------------------------------------------------------------------
# TC-01: Baseline happy path
# ---------------------------------------------------------------------------

def test_TC01_baseline_happy_path():
    """TC-01: the exact scenario from the Account ABC spreadsheet."""
    results = calculate_rebalance(baseline_portfolio(), total_assets=100_000)

    assert_shares_match(results, {"IBM": 67, "MSFT": 0, "ORCL": -45, "AAPL": 0, "HD": 0})

    by_symbol = {r.symbol: r for r in results}
    assert by_symbol["IBM"].shares_raw == pytest.approx(66.6667, abs=0.001)
    assert by_symbol["IBM"].action == "BUY"
    assert by_symbol["ORCL"].shares_raw == pytest.approx(-45.4545, abs=0.001)
    assert by_symbol["ORCL"].action == "SELL"
    for sym in ("MSFT", "AAPL", "HD"):
        assert by_symbol[sym].action == "HOLD"

    # total dollars bought vs sold should roughly offset (rounding drift check, see TC-15)
    total_buy = sum(r.shares_rounded * r.unit_price for r in results if r.shares_rounded > 0)
    total_sell = sum(-r.shares_rounded * r.unit_price for r in results if r.shares_rounded < 0)
    assert abs(total_buy - total_sell) < 1000  # generous tolerance for a 2-line rebalance


# ---------------------------------------------------------------------------
# TC-02: Already on target -> no trades
# ---------------------------------------------------------------------------

def test_TC02_all_on_target_produces_no_trades():
    portfolio = [
        SecurityInput(symbol="IBM", target_pct=20, current_pct=20, unit_price=150),
        SecurityInput(symbol="MSFT", target_pct=20, current_pct=20, unit_price=90),
        SecurityInput(symbol="ORCL", target_pct=20, current_pct=20, unit_price=220),
        SecurityInput(symbol="AAPL", target_pct=20, current_pct=20, unit_price=450),
        SecurityInput(symbol="HD", target_pct=20, current_pct=20, unit_price=70),
    ]
    results = calculate_rebalance(portfolio, total_assets=100_000)
    assert all(r.shares_rounded == 0 and r.action == "HOLD" for r in results)


# ---------------------------------------------------------------------------
# TC-03: Fully underweight single security
# ---------------------------------------------------------------------------

def test_TC03_single_security_fully_underweight():
    portfolio = [SecurityInput(symbol="IBM", target_pct=100, current_pct=0, unit_price=150)]
    results = calculate_rebalance(portfolio, total_assets=100_000)
    assert results[0].action == "BUY"
    assert results[0].shares_rounded == round(100_000 / 150)


# ---------------------------------------------------------------------------
# TC-04: total_assets = 0 -> must not divide by zero / must raise cleanly
# ---------------------------------------------------------------------------

def test_TC04_zero_total_assets_raises_value_error():
    with pytest.raises(ValueError):
        calculate_rebalance(baseline_portfolio(), total_assets=0)


def test_TC04b_negative_total_assets_raises_value_error():
    with pytest.raises(ValueError):
        calculate_rebalance(baseline_portfolio(), total_assets=-100_000)


# ---------------------------------------------------------------------------
# TC-05 / TC-06: bad price data
# ---------------------------------------------------------------------------

def test_TC05_zero_unit_price_raises_value_error():
    portfolio = [SecurityInput(symbol="IBM", target_pct=20, current_pct=10, unit_price=0)]
    with pytest.raises(ValueError):
        calculate_rebalance(portfolio, total_assets=100_000)


def test_TC06_negative_unit_price_raises_value_error():
    portfolio = [SecurityInput(symbol="IBM", target_pct=20, current_pct=10, unit_price=-150)]
    with pytest.raises(ValueError):
        calculate_rebalance(portfolio, total_assets=100_000)


# ---------------------------------------------------------------------------
# TC-07 / TC-08: allocation integrity (does NOT raise -- flags via helper)
# ---------------------------------------------------------------------------

def test_TC07_target_percentages_not_summing_to_100_is_flagged_not_fatal():
    portfolio = [
        SecurityInput(symbol="IBM", target_pct=30, current_pct=10, unit_price=150),
        SecurityInput(symbol="MSFT", target_pct=30, current_pct=20, unit_price=90),
        SecurityInput(symbol="ORCL", target_pct=30, current_pct=30, unit_price=220),
        # targets sum to 90, not 100
    ]
    check = validate_allocations(portfolio)
    assert check["target_sums_to_100"] is False
    # engine still computes a result per line rather than crashing
    results = calculate_rebalance(portfolio, total_assets=100_000)
    assert len(results) == 3


def test_TC08_current_percentages_not_summing_to_100_is_flagged():
    portfolio = [
        SecurityInput(symbol="IBM", target_pct=20, current_pct=10, unit_price=150),
        SecurityInput(symbol="MSFT", target_pct=20, current_pct=10, unit_price=90),
        # current sums to only 20
    ]
    check = validate_allocations(portfolio)
    assert check["current_sums_to_100"] is False


# ---------------------------------------------------------------------------
# TC-09: sub-share variance rounds to 0 (no phantom trade)
# ---------------------------------------------------------------------------

def test_TC09_sub_share_variance_rounds_to_zero():
    # $5 diff on a $450 stock -> 0.011 shares -> rounds to 0, action HOLD
    portfolio = [SecurityInput(symbol="AAPL", target_pct=20.005, current_pct=20, unit_price=450)]
    results = calculate_rebalance(portfolio, total_assets=100_000)
    assert results[0].shares_rounded == 0
    assert results[0].action == "HOLD"
    # but the raw variance is NOT silently zero -- it's just below one share
    assert results[0].shares_raw != 0


# ---------------------------------------------------------------------------
# TC-10: rounding boundary behaviour is explicit and documented
# ---------------------------------------------------------------------------

@pytest.mark.parametrize("current,target,price,expected_shares", [
    (10, 20, 150, 67),   # 66.667 -> rounds up to 67
    (30, 20, 220, -45),  # -45.4545 -> rounds to -45 (0.4545 rounds down)
    (0, 5, 100, 50),     # exact half-integer-free case, sanity check
])
def test_TC10_rounding_boundaries(current, target, price, expected_shares):
    shares = calculate_single_line(current_pct=current, target_pct=target, unit_price=price, total_assets=100_000)
    assert shares == expected_shares


# ---------------------------------------------------------------------------
# TC-11: all positions overweight -> all sells, no buys (no self-funding assumption)
# ---------------------------------------------------------------------------

def test_TC11_all_overweight_produces_only_sells():
    portfolio = [
        SecurityInput(symbol="IBM", target_pct=10, current_pct=20, unit_price=150),
        SecurityInput(symbol="MSFT", target_pct=10, current_pct=20, unit_price=90),
    ]
    results = calculate_rebalance(portfolio, total_assets=100_000)
    assert all(r.action == "SELL" for r in results)
    assert not any(r.action == "BUY" for r in results)


# ---------------------------------------------------------------------------
# TC-12: negative percentages rejected
# ---------------------------------------------------------------------------

def test_TC12_negative_percentage_raises_value_error():
    portfolio = [SecurityInput(symbol="IBM", target_pct=-5, current_pct=10, unit_price=150)]
    with pytest.raises(ValueError):
        calculate_rebalance(portfolio, total_assets=100_000)


# ---------------------------------------------------------------------------
# TC-13: empty portfolio handled gracefully
# ---------------------------------------------------------------------------

def test_TC13_empty_portfolio_returns_empty_list():
    assert calculate_rebalance([], total_assets=100_000) == []


# ---------------------------------------------------------------------------
# TC-14: large-scale account, precision holds
# ---------------------------------------------------------------------------

def test_TC14_large_scale_account_precision():
    portfolio = [
        SecurityInput(symbol="IBM", target_pct=20, current_pct=10, unit_price=150.37),
        SecurityInput(symbol="ORCL", target_pct=20, current_pct=30, unit_price=220.81),
    ]
    results = calculate_rebalance(portfolio, total_assets=50_000_000)
    by_symbol = {r.symbol: r for r in results}
    assert by_symbol["IBM"].action == "BUY"
    assert by_symbol["ORCL"].action == "SELL"
    # no overflow / NaN
    for r in results:
        assert r.shares_rounded == r.shares_rounded  # NaN check (NaN != NaN)


# ---------------------------------------------------------------------------
# TC-15: rounding drift across a multi-security portfolio stays within tolerance
# ---------------------------------------------------------------------------

def test_TC15_rounding_drift_within_tolerance():
    results = calculate_rebalance(baseline_portfolio(), total_assets=100_000)
    total_buy_dollars = sum(r.shares_rounded * r.unit_price for r in results if r.shares_rounded > 0)
    total_sell_dollars = abs(sum(r.shares_rounded * r.unit_price for r in results if r.shares_rounded < 0))
    drift = abs(total_buy_dollars - total_sell_dollars)
    assert drift < 500  # rounding to whole shares on $10k trades should drift far less than this


# ---------------------------------------------------------------------------
# Property-style check: variance sign always matches action
# ---------------------------------------------------------------------------

@pytest.mark.parametrize("current,target", [(10, 20), (30, 20), (20, 20), (0, 100), (100, 0)])
def test_variance_sign_matches_action(current, target):
    portfolio = [SecurityInput(symbol="X", target_pct=target, current_pct=current, unit_price=100)]
    result = calculate_rebalance(portfolio, total_assets=100_000)[0]
    if result.variance_pct < 0:
        assert result.action in ("BUY", "HOLD")  # HOLD only if rounds to 0
    elif result.variance_pct > 0:
        assert result.action in ("SELL", "HOLD")
    else:
        assert result.action == "HOLD"


# ---------------------------------------------------------------------------
# TC-19: full portfolio where every security needs a trade
# ---------------------------------------------------------------------------

def test_TC19_full_portfolio_all_lines_trade():
    portfolio = [
        SecurityInput(symbol="IBM", target_pct=20, current_pct=15, unit_price=150),
        SecurityInput(symbol="MSFT", target_pct=20, current_pct=28, unit_price=90),
        SecurityInput(symbol="ORCL", target_pct=20, current_pct=10, unit_price=220),
        SecurityInput(symbol="AAPL", target_pct=20, current_pct=23, unit_price=450),
        SecurityInput(symbol="HD", target_pct=20, current_pct=24, unit_price=70),
    ]
    results = calculate_rebalance(portfolio, total_assets=100_000)
    assert_shares_match(results, {"IBM": 33, "MSFT": -89, "ORCL": 45, "AAPL": -7, "HD": -57})


# ---------------------------------------------------------------------------
# TC-20: same baseline % structure at a different total asset value
# ---------------------------------------------------------------------------

def test_TC20_scale_invariance():
    """Same %/price structure as TC-01, but total assets = $250,000 instead
    of $100,000. Actions must match TC-01 exactly; only the magnitudes scale."""
    baseline_results = calculate_rebalance(baseline_portfolio(), total_assets=100_000)
    scaled_results = calculate_rebalance(baseline_portfolio(), total_assets=250_000)

    assert_shares_match(scaled_results, {"IBM": 167, "MSFT": 0, "ORCL": -114, "AAPL": 0, "HD": 0})

    baseline_actions = {r.symbol: r.action for r in baseline_results}
    scaled_actions = {r.symbol: r.action for r in scaled_results}
    assert baseline_actions == scaled_actions  # not hardcoded to $100,000


# ---------------------------------------------------------------------------
# TC-25: partial vesting reduces the investable base
# ---------------------------------------------------------------------------

def test_TC25_partial_vesting_reduces_investable_base():
    """80% vested on a $100,000 account -> only $80,000 is investable.
    Design decision (see rebalancer.py assumption #7): target_pct applies
    against the vested base, not the full account value. Flag for business
    sign-off before relying on this in production."""
    portfolio = [SecurityInput(symbol="IBM", target_pct=100, current_pct=0, unit_price=150)]
    results = calculate_rebalance(portfolio, total_assets=100_000, vested_pct=80)

    assert results[0].investable_base == pytest.approx(80_000)
    assert results[0].shares_rounded == 533  # $80,000 / $150 = 533.33 -> 533
    assert results[0].action == "BUY"


def test_TC25b_full_vesting_is_the_default_and_backward_compatible():
    """vested_pct defaults to 100 -> identical to every pre-vesting test."""
    with_default = calculate_rebalance(baseline_portfolio(), total_assets=100_000)
    with_explicit_100 = calculate_rebalance(baseline_portfolio(), total_assets=100_000, vested_pct=100)
    assert [r.shares_rounded for r in with_default] == [r.shares_rounded for r in with_explicit_100]


def test_TC25c_invalid_vested_pct_raises_value_error():
    portfolio = [SecurityInput(symbol="IBM", target_pct=100, current_pct=0, unit_price=150)]
    with pytest.raises(ValueError):
        calculate_rebalance(portfolio, total_assets=100_000, vested_pct=0)
    with pytest.raises(ValueError):
        calculate_rebalance(portfolio, total_assets=100_000, vested_pct=150)



Run the suite:

In [13]:
ipytest.run('-vv')

======================================= test session starts ========================================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.11.1, typeguard-4.6.0, anyio-4.14.2
collecting ... collected 28 items

t_9a25abede74c433e8b899b8f29e2a630.py::test_TC01_baseline_happy_path PASSED                  [  3%]
t_9a25abede74c433e8b899b8f29e2a630.py::test_TC02_all_on_target_produces_no_trades PASSED     [  7%]
t_9a25abede74c433e8b899b8f29e2a630.py::test_TC03_single_security_fully_underweight PASSED    [ 10%]
t_9a25abede74c433e8b899b8f29e2a630.py::test_TC04_zero_total_assets_raises_value_error PASSED [ 14%]
t_9a25abede74c433e8b899b8f29e2a630.py::test_TC04b_negative_total_assets_raises_value_error PASSED [ 17%]
t_9a25abede74c433e8b899b8f29e2a630.py::test_TC05_zero_unit_price_raises_value_error PASSED   [ 21%]
t_9a25abede74c433e8b899b8f29e2a630.py::test_TC06_negative_unit_price_raises_val

<ExitCode.OK: 0>

## 6. Conclusion

This notebook answers one concrete question, for Account ABC's portfolio,
exactly how many shares of each security to buy or sell to bring every
position back to its target allocation  and then proves the answer is
correct.